# Layer-wise MI/SMI Training Analysis

This notebook launches and inspects the Python pipeline for the generalization/layer-wise benchmark setup.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "Generalization and Memorization":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "notebook_examples"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(PROJECT_ROOT)

## Run a small layer-wise example

The full defaults use `m=500` projections and `n=10000` training samples. This example uses smaller values for quick interactive checks.

In [ ]:
cmd = [
    sys.executable,
    "-m", "experiments.run_layer_analysis",
    "--setup", "generalization",
    "--dataset", "mnist",
    "--model", "mlp5",
    "--estimator", "smi_ksg_cd",
    "--epochs", "1",
    "--analysis-epochs", "1",
    "--seeds", "0",
    "--layers", "fc1,fc2,fc3,fc4",
    "--max-mi-samples", "256",
    "--n-projs", "8",
    "--n-jobs", "-1",
    "--output-dir", str(OUTPUT_DIR),
    "--no-checkpoints",
]
print(" ".join(cmd))
# subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

## Load layer-wise estimates

In [ ]:
results_path = OUTPUT_DIR / "layer_analysis" / "mnist" / "mlp5" / "smi_ksg_cd" / "results.csv"
if results_path.exists():
    results = pd.read_csv(results_path)
    display(results.head())
else:
    print(f"Run the command above first. Missing: {results_path}")

## Aggregate and plot layer comparisons

In [ ]:
aggregate_path = OUTPUT_DIR / "layer_analysis_summary.csv"
aggregate_cmd = [sys.executable, "-m", "experiments.aggregate_results", "--input", str(results_path), "--output", str(aggregate_path)]
plot_cmd = [sys.executable, "-m", "experiments.plot_results", "--input", str(aggregate_path), "--output-dir", str(OUTPUT_DIR / "figures"), "--x", "epoch", "--hue", "layer"]
print(" ".join(aggregate_cmd))
print(" ".join(plot_cmd))
# subprocess.run(aggregate_cmd, cwd=PROJECT_ROOT, check=True)
# subprocess.run(plot_cmd, cwd=PROJECT_ROOT, check=True)

## Direct feature-extraction example

In [ ]:
from experiments.datasets import load_dataset
from experiments.models import build_model, default_analysis_layers
from experiments.features import extract_activations
from experiments.train import compile_model
from experiments.utils import sample_indices, set_global_seed

set_global_seed(0)
data = load_dataset("mnist")
model = compile_model(build_model("mlp5", data.input_shape, data.num_classes), learning_rate=1e-2, momentum=0.9)
layers = default_analysis_layers(model)
idx = sample_indices(len(data.x_train), max_items=32, seed=0)
activations = extract_activations(model, data.x_train[idx], layers, batch_size=32)
print(layers)
print({name: value.shape for name, value in activations.items()})